# Basel III Capital Adequacy

**Supervisory stress-test failure analysis for bank capital resilience**

Regulatory framework: Basel III (BCBS), CCAR / DFAST (Federal Reserve),
EBA stress-testing guidelines, SR 11-7 model risk management.

## Part 0 — Setup

In [3]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, precision_recall_fscore_support)
warnings.filterwarnings('ignore')
from hugiml import HUGIMLClassifierNative
from hugiml.calibration import evaluate_calibration
from hugiml.metrics import compute_all_metrics
from hugiml.pruning import PatternEditor
from hugiml.governance import generate_model_card

RANDOM_STATE = 42
DATA_FILE     = 'nb07_basel_ca_data.csv'
METADATA_FILE = 'nb07_basel_ca_metadata.csv'
TARGET = 'fails_stress_test'

BASEL_MAP = {
    'cet1_ratio':'CET1 Shortfall','tier1_ratio':'Tier 1 Capital Adequacy',
    'total_capital_ratio':'Total Capital Ratio','leverage_ratio':'Leverage Ratio',
    'rwa_density':'RWA Density','loan_to_deposit_ratio':'Funding Pressure',
    'npl_ratio':'Asset Quality / NPL Stress','concentration_risk':'Concentration Risk',
    'liquidity_coverage_ratio':'LCR Liquidity Stress',
    'net_stable_funding_ratio':'NSFR Structural Funding',
    'market_risk_rwa_pct':'Market Risk','operational_risk_rwa_pct':'Operational Risk',
    'bank_size':'Bank Size','business_model':'Business Model',
}

import hugiml; print('hugiml-core', hugiml.__version__)


hugiml-core 1.1.2


## Part 1 — Data Loading

In [5]:
df = pd.read_csv(DATA_FILE)
metadata = pd.read_csv(METADATA_FILE)
df[TARGET] = df[TARGET].astype(int)
X = df.drop(columns=[TARGET]); y = df[TARGET]
print(f'Banks: {len(df):,} | Features: {df.shape[1]-1} | Failure rate: {y.mean():.2%}')
print(f'bank_size: {df.bank_size.value_counts().to_dict()}')
print(f'business_model: {df.business_model.value_counts().to_dict()}')
print('\nFeature register (top 10):')
print(metadata[['column_name','importance','notes']].head(10).to_string(index=False))


Banks: 5,000 | Features: 15 | Failure rate: 8.00%
bank_size: {'medium': 1877, 'small': 1650, 'large': 1473}
business_model: {'retail': 2090, 'commercial': 1761, 'universal': 1149}

Feature register (top 10):
             column_name importance                                                                     notes
              cet1_ratio   Critical      Basel III minimum: 4.5% + 2.5% buffer = 7.0%; <10% is risk indicator
             tier1_ratio       High                              Basel III minimum: 6.0%; includes CET1 + AT1
     total_capital_ratio       High                         Basel III minimum: 8.0%; includes Tier 1 + Tier 2
          leverage_ratio   Critical              Basel III minimum: 3.0%; <4.6% indicates undercapitalization
             rwa_density     Medium                      Higher RWA density indicates riskier asset portfolio
   loan_to_deposit_ratio     Medium                                    Ratios >100% indicate funding pressure
               npl_rat

## Part 2 — Splits (60/20/20)

> **CCAR/DFAST validation best practice:** Train on historical stress cycles,
> test on the most recent cycle (out-of-cycle validation). Without explicit cycle
> identifiers here we use a stratified split with a dedicated calibration holdout
> for isotonic recalibration of failure probabilities used in capital buffer planning.


In [7]:
clf_prep = HUGIMLClassifierNative(B=10, L=1, G=5e-4, topK=100)
X_enc, y_enc = clf_prep.prepareXy(X, y)
X_tr,X_tmp,y_tr,y_tmp = train_test_split(X_enc,y_enc,test_size=0.40,stratify=y_enc,random_state=RANDOM_STATE)
X_cal,X_te,y_cal,y_te = train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=RANDOM_STATE)
print(f'Train:{len(X_tr):,} (fail:{y_tr.sum()})  Cal:{len(X_cal):,}  Test:{len(X_te):,} (fail:{y_te.sum()})')


Train:3,000 (fail:240)  Cal:1,000  Test:1,000 (fail:80)


## Part 3 — feature_mode Comparison

In [9]:
mode_res={}
for mode in ['patterns_only','original_plus_patterns']:
    c=HUGIMLClassifierNative(B=10,L=1,G=5e-4,topK=100,feature_mode=mode)
    Xe,ye=c.prepareXy(X,y)
    Xtr2,Xtmp2,ytr2,ytmp2=train_test_split(Xe,ye,test_size=0.40,stratify=ye,random_state=RANDOM_STATE)
    Xcal2,Xte2,ycal2,yte2=train_test_split(Xtmp2,ytmp2,test_size=0.50,stratify=ytmp2,random_state=RANDOM_STATE)
    c.fit(Xtr2,ytr2); p2=c.predict_proba(Xte2)[:,1]
    mode_res[mode]=dict(clf=c,Xtr=Xtr2,Xcal=Xcal2,Xte=Xte2,ytr=ytr2,ycal=ycal2,yte=yte2,proba=p2,
                        auc=roc_auc_score(yte2,p2),ap=average_precision_score(yte2,p2))
    print(f"  {mode:26s}: {len(c.get_hug_features()):3d} patterns | AUC={mode_res[mode]['auc']:.4f} | AP={mode_res[mode]['ap']:.4f}")

R=mode_res['patterns_only']
clf,X_tr,X_cal,X_te=R['clf'],R['Xtr'],R['Xcal'],R['Xte']
y_tr,y_cal,y_te,y_score=R['ytr'],R['ycal'],R['yte'],R['proba']
auc,ap=R['auc'],R['ap']
fpr_c,tpr_c,thr_roc=roc_curve(y_te,y_score)
youden_idx=int(np.argmax(tpr_c-fpr_c))
op_thr=float(thr_roc[youden_idx]) if np.isfinite(thr_roc[youden_idx]) else 0.50
y_pred=(y_score>=op_thr).astype(int)
tn,fp,fn,tp=confusion_matrix(y_te,y_pred).ravel()
prec_op,rec_op,f1_op,_=precision_recall_fscore_support(y_te,y_pred,average='binary',zero_division=0)
print(f'Baseline @ Youden {op_thr:.3f}: TP={tp}  FP={fp}  TN={tn}  FN={fn}  precision={prec_op:.2%}  recall={rec_op:.2%}')


  patterns_only             :  71 patterns | AUC=0.9455 | AP=0.7549
  original_plus_patterns    :  71 patterns | AUC=0.9541 | AP=0.7668
Baseline @ Youden 0.049: TP=74  FP=180  TN=740  FN=6  precision=29.13%  recall=92.50%


## Part 4 — Calibration

In [11]:
cal_pre = evaluate_calibration(np.asarray(y_te), y_score)
print(f'ECE={cal_pre.ece:.4f}  MCE={cal_pre.mce:.4f}  Brier={cal_pre.brier_score:.4f}')


ECE=0.0122  MCE=0.2415  Brier=0.0357


## Part 5 — Interpretability Metrics

In [13]:
interp = compute_all_metrics(clf, X_te)
print(f'n_patterns:{interp.n_patterns}  coverage:{interp.coverage:.2%}  mean_active:{interp.mean_active_patterns:.2f}')


n_patterns:71  coverage:100.00%  mean_active:7.65


## Part 6 — Basel III Regulatory Pattern Mapping

Each pattern is labelled with its corresponding Basel III capital pillar:
Pillar 1 (minimum capital), Pillar 2 (supervisory review), Pillar 3 (disclosure).


In [15]:
importances = clf.feature_importances().copy()
def _lbl(p):
    for kw,l in BASEL_MAP.items():
        if kw in p: return l
    return 'Other'
importances['regulatory_driver'] = importances['pattern'].apply(_lbl)
importances['direction'] = np.where(importances['coefficient']>=0,'failure-signal','strength-signal')
top15 = importances.nlargest(15,'abs_coefficient')
print('Top 15 patterns with Basel III regulatory drivers:')
print('='*90)
for _,row in top15.iterrows():
    a='▲' if row['coefficient']>0 else '▼'
    print(f"  {a} {row['pattern']:45s} {row['coefficient']:+.4f}  sup={row['support']:.1%}  [{row['regulatory_driver']}]")
print('='*90)
print('\nPatterns per driver:'); print(importances.groupby('regulatory_driver').size().sort_values(ascending=False))


Top 15 patterns with Basel III regulatory drivers:
  ▲ liquidity_coverage_ratio=[88,95)              +4.4710  sup=1.7%  [LCR Liquidity Stress]
  ▲ net_stable_funding_ratio=[94,98)              +4.1286  sup=2.1%  [NSFR Structural Funding]
  ▼ operational_risk_rwa_pct=[7,9)                -2.5422  sup=19.6%  [Operational Risk]
  ▼ rwa_density=[38,42.72)                        -2.0182  sup=10.0%  [RWA Density]
  ▼ loan_to_deposit_ratio=[55,61.88)              -1.9465  sup=10.0%  [Funding Pressure]
  ▼ concentration_risk=[7,9.717)                  -1.8211  sup=10.0%  [Concentration Risk]
  ▼ market_risk_rwa_pct=[3,3.704)                 -1.8031  sup=10.0%  [Market Risk]
  ▼ leverage_ratio=[7.114,9.286)                  -1.1910  sup=10.0%  [Leverage Ratio]
  ▼ npl_ratio=[1.263,1.695)                       -1.1686  sup=10.0%  [Asset Quality / NPL Stress]
  ▼ net_stable_funding_ratio=[98,100.2)           -1.0970  sup=17.9%  [NSFR Structural Funding]
  ▲ npl_ratio=[4.453,8.775)                

## Part 7 — Pattern Review (SR 11-7 / CCAR)

SR 11-7 requires documented rationale for all model changes. The PatternEditor
provides the audit trail for model validation packages.

**Key decision:** Patterns for LCR=[88,95%) and NSFR=[94,98%) have low support
(~1.7–2%) but very high coefficients (+4.5, +4.1). These are **retained** as they
represent genuine tail-risk stress indicators — banks at the regulatory minimum.
Only patterns that are BOTH low-support AND low-coefficient are treated as noise.


In [17]:
editor = PatternEditor(clf, operator_name='model-validation-sr117')
pats_df = editor.list_patterns()
print(f'Patterns before review: {len(pats_df)}')

# Remove only LOW-support AND LOW-coefficient patterns (noise)
noise_pats = pats_df[
    (pats_df['support'] < 0.03) &
    (pats_df['coefficient'].abs() < 0.50)
]
print(f'Noise patterns (low support + low coefficient): {len(noise_pats)}')
if len(noise_pats):
    editor.remove(noise_pats['idx'].tolist(),
        reason='Low-support AND low-coefficient patterns: noise unstable across stress vintages; '
               'high-coefficient LCR/NSFR patterns retained as primary Basel III stress indicators')

editor.refit(X_tr, y_tr)
editor.calibrate(X_cal, y_cal, method='isotonic')
clf_pruned = editor.finalize()

proba_pruned = clf_pruned.predict_proba(X_te)[:,1]
auc_pruned = roc_auc_score(y_te, proba_pruned)
ap_pruned  = average_precision_score(y_te, proba_pruned)
cal_post   = evaluate_calibration(np.asarray(y_te), proba_pruned)
fpr_p,tpr_p,thr_p=roc_curve(y_te,proba_pruned)
op_thr_p=float(thr_p[int(np.argmax(tpr_p-fpr_p))])
y_pred_p=(proba_pruned>=op_thr_p).astype(int)
tn_p,fp_p,fn_p,tp_p=confusion_matrix(y_te,y_pred_p).ravel()
prec_p,rec_p,f1_p,_=precision_recall_fscore_support(y_te,y_pred_p,average='binary',zero_division=0)
print(f'After isotonic recalibration: AUC={auc_pruned:.4f}  AP={ap_pruned:.4f}  ECE={cal_post.ece:.4f}')
print(f'Precision={prec_p:.2%}  Recall={rec_p:.2%}  F1={f1_p:.3f}')
audit_js=json.loads(editor.audit_report())
print(f'Audit: removed={audit_js["diff"]["n_removed"]}, calibrated={audit_js["calibration"]["applied"]}')


Patterns before review: 71
Noise patterns (low support + low coefficient): 0
After isotonic recalibration: AUC=0.9229  AP=0.6733  ECE=0.0118
Precision=46.88%  Recall=75.00%  F1=0.577
Audit: removed=0, calibrated=True


## Part 8 — Capital Buffer Threshold Analysis

In [19]:
thresh_rows=[]
for thr in np.linspace(0.05,0.95,37):
    pred=(proba_pruned>=thr).astype(int)
    if pred.sum()==0: continue
    tn_t,fp_t,fn_t,tp_t=confusion_matrix(np.asarray(y_te),pred,labels=[0,1]).ravel()
    prec_t,rec_t,f1_t,_=precision_recall_fscore_support(y_te,pred,average='binary',zero_division=0)
    thresh_rows.append({'threshold':thr,'n_flagged':int(pred.sum()),
                        'tp':tp_t,'fp':fp_t,'fn':fn_t,'precision':prec_t,'recall':rec_t,'f1':f1_t})
threshold_table=pd.DataFrame(thresh_rows)
print('Capital buffer operating points:')
for tr in [0.70,0.80,0.90]:
    r=threshold_table[threshold_table['recall']>=tr]
    if len(r):
        row=r.iloc[-1]
        print(f"  {int(tr*100)}% recall: thr={row.threshold:.3f}  precision={row.precision:.2%}  flags={int(row.n_flagged)}")
threshold_table.round(4)


Capital buffer operating points:
  70% recall: thr=0.125  precision=54.90%  flags=102
  80% recall: thr=0.075  precision=26.91%  flags=249
  90% recall: thr=0.050  precision=21.33%  flags=347


,threshold,n_flagged,tp,fp,fn,precision,recall,f1
0,0.050,347,74,273,6,0.2133,0.9250,0.3466
1,0.075,249,67,182,13,0.2691,0.8375,0.4073
2,0.100,130,60,70,20,0.4615,0.7500,0.5714
3,0.125,102,56,46,24,0.5490,0.7000,0.6154
4,0.150,85,51,34,29,0.6000,0.6375,0.6182
5,0.175,85,51,34,29,0.6000,0.6375,0.6182
6,0.200,85,51,34,29,0.6000,0.6375,0.6182
7,0.225,85,51,34,29,0.6000,0.6375,0.6182
8,0.250,85,51,34,29,0.6000,0.6375,0.6182
9,0.275,84,51,33,29,0.6071,0.6375,0.6220


## Part 9 — Subgroup Audit

In [21]:
test_idx=X_te.index if hasattr(X_te,'index') else pd.Index(range(len(y_te)))
raw_test=X.loc[test_idx].copy()
af=raw_test.copy(); af['_actual']=np.asarray(y_te).astype(int)
af['_score']=proba_pruned; af['_flag']=y_pred_p
sub_rows=[]
for col in ['bank_size','business_model']:
    if col not in af.columns: continue
    for level in af[col].value_counts().index:
        g=af[af[col].eq(level)]
        if len(g)<20: continue
        yg=g['_actual'].to_numpy(); fg=g['_flag'].to_numpy()
        auc_g=roc_auc_score(yg,g['_score']) if len(np.unique(yg))==2 else np.nan
        tn_g,fp_g,fn_g,tp_g=confusion_matrix(yg,fg,labels=[0,1]).ravel()
        sub_rows.append({'feature':col,'segment':str(level),'n':len(g),
            'base_rate':yg.mean(),'flag_rate':fg.mean(),'auc':auc_g,
            'precision':tp_g/max(tp_g+fp_g,1),'recall':tp_g/max(tp_g+fn_g,1)})
subgroup_audit=pd.DataFrame(sub_rows)
subgroup_audit.sort_values(['feature','n'],ascending=[True,False]).round(4)


,feature,segment,n,base_rate,flag_rate,auc,precision,recall
0,bank_size,medium,397,0.0957,0.1436,0.9087,0.4912,0.7368
1,bank_size,large,305,0.0754,0.1344,0.9611,0.4878,0.8696
2,bank_size,small,298,0.0638,0.1007,0.9014,0.4000,0.6316
3,business_model,retail,419,0.0955,0.1384,0.9173,0.5000,0.7250
4,business_model,commercial,357,0.0644,0.1261,0.9225,0.3778,0.7391
5,business_model,universal,224,0.0759,0.1116,0.9305,0.5600,0.8235


## Part 10 — Decile Analysis

In [23]:
score_df=pd.DataFrame({'actual':np.asarray(y_te).astype(int),'score':proba_pruned})
score_df['decile']=pd.qcut(score_df['score'].rank(method='first'),10,labels=range(1,11)).astype(int)
decile=score_df.groupby('decile').agg(n=('actual','size'),event_rate=('actual','mean'),
    events=('actual','sum')).reset_index()
decile['capture_pct']=decile['events']/max(decile['events'].sum(),1)
decile=decile.sort_values('decile',ascending=False)
base=float(score_df['actual'].mean())
top=float(decile[decile['decile'].eq(10)]['event_rate'].iloc[0])
print(f'Top decile failure rate: {top:.2%}  ({top/base:.1f}× base rate)')
decile.round(4)


Top decile failure rate: 55.00%  (6.9× base rate)


,decile,n,event_rate,events,capture_pct
9,10,100,0.55,55,0.6875
8,9,100,0.08,8,0.1000
7,8,100,0.09,9,0.1125
6,7,100,0.02,2,0.0250
5,6,100,0.05,5,0.0625
4,5,100,0.01,1,0.0125
3,4,100,0.00,0,0.0000
2,3,100,0.00,0,0.0000
1,2,100,0.00,0,0.0000
0,1,100,0.00,0,0.0000


## Part 11 — Covariate Drift (PSI)

In [25]:
def compute_psi(expected,actual,buckets=10):
    rows=[]
    common=expected.select_dtypes(include=[np.number]).columns.intersection(actual.columns)
    for col in common:
        edges=np.percentile(expected[col].dropna(),np.linspace(0,100,buckets+1))
        edges[0]=-np.inf; edges[-1]=np.inf
        ep=np.maximum(np.histogram(expected[col],bins=edges)[0]/len(expected),1e-6)
        ap_=np.maximum(np.histogram(actual[col],bins=edges)[0]/len(actual),1e-6)
        psi=float(np.sum((ap_-ep)*np.log(ap_/ep)))
        rows.append({'feature':col,'psi':round(psi,4),
            'status':'STABLE' if psi<0.10 else 'WARNING' if psi<0.25 else 'SHIFT'})
    return pd.DataFrame(rows).sort_values('psi',ascending=False)

rng_d=np.random.default_rng(13); n_d=500
num_cols7=X.select_dtypes(include=[np.number]).columns.tolist()
X_raw_train=X[num_cols7].iloc[:3000]
# Macro-stress shift: depressed capital ratios, elevated NPL/LCR stress
X_raw_drift=pd.DataFrame({'cet1_ratio':rng_d.uniform(7.0,9.5,n_d),
    'leverage_ratio':rng_d.uniform(3.0,4.5,n_d),'npl_ratio':rng_d.uniform(5.0,10.0,n_d),
    'liquidity_coverage_ratio':rng_d.uniform(88.0,102.0,n_d),
    'rwa_density':rng_d.uniform(60.0,86.0,n_d)})
X_raw_drift=X_raw_drift[[c for c in X_raw_drift.columns if c in X_raw_train.columns]]
psi_df=compute_psi(X_raw_train,X_raw_drift)
print('PSI vs simulated macro-stress window:'); print(psi_df.to_string(index=False))


PSI vs simulated macro-stress window:
                 feature     psi status
              cet1_ratio 12.4339  SHIFT
          leverage_ratio 12.4339  SHIFT
               npl_ratio 12.4339  SHIFT
liquidity_coverage_ratio 11.1690  SHIFT
             rwa_density  6.2547  SHIFT


## Part 12 — Model Governance (SR 11-7)

In [27]:
card = generate_model_card(
    clf_pruned, model_id='basel-stress-test-v1.0',
    intended_use='Predicting CCAR/DFAST stress-test failure probability for capital buffer planning.',
    out_of_scope_use='Not for individual credit decisions. Not a substitute for official supervisory calculations.',
    training_data_description=f'Synthetic Basel III dataset, {len(df):,} banks, 60/20/20 split.',
    evaluation_data_description=f'Stratified 20% holdout; {int(y_te.sum())} failing banks.',
    performance_metrics={'AUC-ROC':round(auc_pruned,4),'AvgPrecision':round(ap_pruned,4),
        'ECE':round(cal_post.ece,4),'Precision_Youden':round(float(prec_p),4),'Recall_Youden':round(float(rec_p),4)},
    limitations=[
        'Trained on synthetic data — production requires validation on actual CCAR/DFAST outcomes.',
        'Out-of-cycle validation required per SR 11-7.',
        'PSI monitoring required semi-annually.',
    ],
    ethical_considerations='SR 11-7 MRM sign-off required. Pattern-level regulatory driver explanations mandatory.'
)
print(card.to_markdown())
card.save('nb07_basel_ca_model_card.json')
editor.save_audit_report('nb07_basel_ca_audit_trail.json')
importances.to_csv('nb07_basel_ca_pattern_inventory.csv',index=False)
threshold_table.to_csv('nb07_basel_ca_threshold_grid.csv',index=False)
subgroup_audit.to_csv('nb07_basel_ca_subgroup_audit.csv',index=False)
print('\n✓ Governance artifacts saved.')


# Model Card: basel-stress-test-v1.0

**Type:** HUGIMLClassifierNative  
**License:** Apache-2.0  
**Created:** 2026-05-29T06:31:25Z  
**Framework:** hugiml-core 1.1.2

## Reference

Krishnamoorthy, S. (2024). Interpretable Classifier Models for Decision Support Using High Utility Gain Patterns. IEEE Access, 12, 126088-126107. DOI: 10.1109/ACCESS.2024.3455563

## Intended Use

Predicting CCAR/DFAST stress-test failure probability for capital buffer planning.

## Out-of-Scope Use

Not for individual credit decisions. Not a substitute for official supervisory calculations.

## Training Data

Synthetic Basel III dataset, 5,000 banks, 60/20/20 split.

## Evaluation Data

Stratified 20% holdout; 80 failing banks.

## Hyperparameters

- **B**: 10
- **L**: 1
- **G**: 0.0005
- **topK**: 100
- **adaptive_binning**: False
- **feature_mode**: patterns_only

## Performance Metrics

- **AUC-ROC**: 0.9229
- **AvgPrecision**: 0.6733
- **ECE**: 0.0118
- **Precision_Youden**: 0.4688
- **Recall_Youden**